In [1]:
from pyspark.sql import SparkSession
import os
import json
from pprint import pprint
warehouse_path = r"C:\iceberg-warehouse"

print(os.listdir(warehouse_path))

['db', 'demo']


In [2]:
spark = SparkSession.builder \
    .appName("IcebergLocal") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    ) \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    ) \
    .config(
        "spark.sql.catalog.local.warehouse",
        "file:///C:/iceberg-warehouse"
    ) \
    .getOrCreate()

In [3]:
spark.sql("""
CREATE TABLE local.demo.schema_demo(
id INT,
name STRING,
age INT
) using iceberg;

""")

DataFrame[]

In [4]:
spark.sql("""
INSERT INTO local.demo.schema_demo VALUES
(1, 'Alice', 25),
(2, 'Bob', 30),
(3, 'Charlie', 35);
""")

DataFrame[]

In [6]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo;
    """
).collect()

[Row(id=1, name='Alice', age=25),
 Row(id=2, name='Bob', age=30),
 Row(id=3, name='Charlie', age=35)]

In [7]:
spark.sql(
    """
DESCRIBE TABLE local.demo.schema_demo;
    """
).collect()

[Row(col_name='id', data_type='int', comment=None),
 Row(col_name='name', data_type='string', comment=None),
 Row(col_name='age', data_type='int', comment=None)]

In [8]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo.history;
    """
).collect()

[Row(made_current_at=datetime.datetime(2026, 9, 13, 18, 40, 31, 703000), snapshot_id=5419803532513947480, parent_id=None, is_current_ancestor=True)]

In [9]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo.snapshots;
    """
).collect()

[Row(committed_at=datetime.datetime(2026, 9, 13, 18, 40, 31, 703000), snapshot_id=5419803532513947480, parent_id=None, operation='append', manifest_list='file:/C:/iceberg-warehouse/demo/schema_demo/metadata/snap-5419803532513947480-1-3752aad6-f81a-4d34-8a99-fbcc5e0e416b.avro', summary={'spark.app.id': 'local-1789304990571', 'changed-partition-count': '1', 'added-data-files': '3', 'total-equality-deletes': '0', 'added-records': '3', 'total-position-deletes': '0', 'added-files-size': '2513', 'total-delete-files': '0', 'total-files-size': '2513', 'total-records': '3', 'total-data-files': '3'})]

In [10]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo.files;
    """
).collect()

[Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00000-0-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', file_format='PARQUET', spec_id=0, record_count=1, file_size_in_bytes=838, column_sizes={1: 36, 2: 41, 3: 36}, value_counts={1: 1, 2: 1, 3: 1}, null_value_counts={1: 0, 2: 0, 3: 0}, nan_value_counts={}, lower_bounds={1: bytearray(b'\x01\x00\x00\x00'), 2: bytearray(b'Alice'), 3: bytearray(b'\x19\x00\x00\x00')}, upper_bounds={1: bytearray(b'\x01\x00\x00\x00'), 2: bytearray(b'Alice'), 3: bytearray(b'\x19\x00\x00\x00')}, key_metadata=None, split_offsets=[4], equality_ids=None, sort_order_id=0, readable_metrics=Row(age=Row(column_size=36, value_count=1, null_value_count=0, nan_value_count=None, lower_bound=25, upper_bound=25), id=Row(column_size=36, value_count=1, null_value_count=0, nan_value_count=None, lower_bound=1, upper_bound=1), name=Row(column_size=41, value_count=1, null_value_count=0, nan_value_count=None, lower_bound='Alice', upper_bound='Alice

In [12]:
spark.sql("""

ALTER TABLE local.demo.schema_demo
ADD COLUMN email STRING;
""")

DataFrame[]

In [13]:
spark.sql(
    """
DESCRIBE TABLE local.demo.schema_demo;
    """
).collect()

[Row(col_name='id', data_type='int', comment=None),
 Row(col_name='name', data_type='string', comment=None),
 Row(col_name='age', data_type='int', comment=None),
 Row(col_name='email', data_type='string', comment=None)]

In [14]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo;
    """
).collect()

[Row(id=1, name='Alice', age=25, email=None),
 Row(id=2, name='Bob', age=30, email=None),
 Row(id=3, name='Charlie', age=35, email=None)]

In [15]:
spark.sql("""
INSERT INTO local.demo.schema_demo VALUES
(4, 'David', 40, 'david@example.com'),
(5, 'Eva', 28, 'eva@example.com');
""")

DataFrame[]

In [16]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo;
    """
).collect()

[Row(id=1, name='Alice', age=25, email=None),
 Row(id=2, name='Bob', age=30, email=None),
 Row(id=3, name='Charlie', age=35, email=None),
 Row(id=4, name='David', age=40, email='david@example.com'),
 Row(id=5, name='Eva', age=28, email='eva@example.com')]

In [17]:
spark.sql("""
ALTER TABLE local.demo.schema_demo
RENAME COLUMN name TO username;
""")

DataFrame[]

In [18]:
spark.sql(
    """
SELECT * FROM local.demo.schema_demo;
    """
).collect()

[Row(id=4, username='David', age=40, email='david@example.com'),
 Row(id=1, username='Alice', age=25, email=None),
 Row(id=5, username='Eva', age=28, email='eva@example.com'),
 Row(id=2, username='Bob', age=30, email=None),
 Row(id=3, username='Charlie', age=35, email=None)]

In [19]:
files = spark.sql("""
SELECT
    content,
    file_path,
    record_count
FROM local.demo.schema_demo.files;
""").collect()

In [20]:
len(files)

5

In [21]:
files

[Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00000-8-f0e53a28-4c2d-47e1-8c8d-8fbd6a06cbc5-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00001-9-f0e53a28-4c2d-47e1-8c8d-8fbd6a06cbc5-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00000-0-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00001-1-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00002-2-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', record_count=1)]

In [27]:
for i in files:
    path  = i.file_path
    print(spark.read.parquet(path).collect())

[Row(id=4, name='David', age=40, email='david@example.com')]
[Row(id=5, name='Eva', age=28, email='eva@example.com')]
[Row(id=1, name='Alice', age=25)]
[Row(id=2, name='Bob', age=30)]
[Row(id=3, name='Charlie', age=35)]


In [28]:
spark.sql("""
ALTER TABLE local.demo.schema_demo
DROP COLUMN age;
""")

DataFrame[]

In [30]:
spark.sql("""
select * from local.demo.schema_demo
""").collect()

[Row(id=4, username='David', email='david@example.com'),
 Row(id=1, username='Alice', email=None),
 Row(id=5, username='Eva', email='eva@example.com'),
 Row(id=2, username='Bob', email=None),
 Row(id=3, username='Charlie', email=None)]

In [31]:
after_files = spark.sql("""
SELECT
    content,
    file_path,
    record_count
FROM local.demo.schema_demo.files;
""").collect()

In [32]:
after_files

[Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00000-8-f0e53a28-4c2d-47e1-8c8d-8fbd6a06cbc5-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00001-9-f0e53a28-4c2d-47e1-8c8d-8fbd6a06cbc5-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00000-0-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00001-1-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', record_count=1),
 Row(content=0, file_path='file:/C:/iceberg-warehouse/demo/schema_demo/data/00002-2-4ef7d284-b419-499f-ae07-81ea866a7b07-0-00001.parquet', record_count=1)]

In [33]:
for i in after_files:
    path  = i.file_path
    print(spark.read.parquet(path).collect())

[Row(id=4, name='David', age=40, email='david@example.com')]
[Row(id=5, name='Eva', age=28, email='eva@example.com')]
[Row(id=1, name='Alice', age=25)]
[Row(id=2, name='Bob', age=30)]
[Row(id=3, name='Charlie', age=35)]


In [36]:
for i in range(len(after_files)):
    if after_files[i]==files[i]:
        print("file path is same")

file path is same
file path is same
file path is same
file path is same
file path is same


In [38]:
spark.sql("""
select * from local.demo.schema_demo.history
""").collect()

[Row(made_current_at=datetime.datetime(2026, 9, 13, 18, 40, 31, 703000), snapshot_id=5419803532513947480, parent_id=None, is_current_ancestor=True),
 Row(made_current_at=datetime.datetime(2026, 9, 13, 18, 44, 13, 273000), snapshot_id=5930307352716642086, parent_id=5419803532513947480, is_current_ancestor=True)]

In [39]:
spark.sql("""
    SELECT *
    FROM local.demo.schema_demo
    VERSION AS OF 5419803532513947480
""").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 35|
+---+-------+---+



In [40]:
spark.sql("""
    SELECT *
    FROM local.demo.schema_demo
    VERSION AS OF 5930307352716642086
""").show()

+---+-------+---+-----------------+
| id|   name|age|            email|
+---+-------+---+-----------------+
|  4|  David| 40|david@example.com|
|  5|    Eva| 28|  eva@example.com|
|  1|  Alice| 25|             NULL|
|  2|    Bob| 30|             NULL|
|  3|Charlie| 35|             NULL|
+---+-------+---+-----------------+



In [41]:
metadata_path = r"C:\iceberg-warehouse\demo\schema_demo\metadata"

In [42]:
import os

metadata_files = [
    f for f in os.listdir(metadata_path)
    if f.endswith(".metadata.json")
]

metadata_files

['v1.metadata.json',
 'v2.metadata.json',
 'v3.metadata.json',
 'v4.metadata.json',
 'v5.metadata.json',
 'v6.metadata.json']

In [43]:
import json
import os

metadata_file = sorted(metadata_files)[-1]

with open(os.path.join(metadata_path, metadata_file), "r") as f:
    metadata = json.load(f)

metadata.keys()

dict_keys(['format-version', 'table-uuid', 'location', 'last-sequence-number', 'last-updated-ms', 'last-column-id', 'current-schema-id', 'schemas', 'default-spec-id', 'partition-specs', 'last-partition-id', 'default-sort-order-id', 'sort-orders', 'properties', 'current-snapshot-id', 'refs', 'snapshots', 'statistics', 'partition-statistics', 'snapshot-log', 'metadata-log'])

In [44]:
for schema in metadata["schemas"]:
    print("SCHEMA ID:", schema["schema-id"])

    for field in schema["fields"]:
        print(
            f"    field-id={field['id']}, "
            f"name={field['name']}, "
            f"type={field['type']}"
        )

    print()

SCHEMA ID: 0
    field-id=1, name=id, type=int
    field-id=2, name=name, type=string
    field-id=3, name=age, type=int

SCHEMA ID: 1
    field-id=1, name=id, type=int
    field-id=2, name=name, type=string
    field-id=3, name=age, type=int
    field-id=4, name=email, type=string

SCHEMA ID: 2
    field-id=1, name=id, type=int
    field-id=2, name=username, type=string
    field-id=3, name=age, type=int
    field-id=4, name=email, type=string

SCHEMA ID: 3
    field-id=1, name=id, type=int
    field-id=2, name=username, type=string
    field-id=4, name=email, type=string



In [46]:
spark.sql(
    """
select * from local.demo.schema_demo.snapshots
    """
).collect()

[Row(committed_at=datetime.datetime(2026, 9, 13, 18, 40, 31, 703000), snapshot_id=5419803532513947480, parent_id=None, operation='append', manifest_list='file:/C:/iceberg-warehouse/demo/schema_demo/metadata/snap-5419803532513947480-1-3752aad6-f81a-4d34-8a99-fbcc5e0e416b.avro', summary={'spark.app.id': 'local-1789304990571', 'changed-partition-count': '1', 'added-data-files': '3', 'total-equality-deletes': '0', 'added-records': '3', 'total-position-deletes': '0', 'added-files-size': '2513', 'total-delete-files': '0', 'total-files-size': '2513', 'total-records': '3', 'total-data-files': '3'}),
 Row(committed_at=datetime.datetime(2026, 9, 13, 18, 44, 13, 273000), snapshot_id=5930307352716642086, parent_id=5419803532513947480, operation='append', manifest_list='file:/C:/iceberg-warehouse/demo/schema_demo/metadata/snap-5930307352716642086-1-1ed07d43-163c-42da-9476-0ab7178a422f.avro', summary={'spark.app.id': 'local-1789304990571', 'changed-partition-count': '1', 'added-data-files': '2', 'to